# TD — Collation automatique avec CollateX

Ce notebook sert de support pour une séance de travail sur la collation automatique. Il part de témoins déjà préparés dans le dépôt et montre deux usages complémentaires :

1. comparer des fichiers texte ou XML-TEI avec CollateX ;
2. travailler sur des témoins TEI lemmatisés, en comparant les lemmes et les catégories grammaticales.

L’objectif est de comprendre ce que fait l’alignement, quels choix méthodologiques interviennent et comment lire les résultats obtenus.


## 0. Préparation

La cellule suivante charge les bibliothèques, définit les chemins du dépôt et rassemble quelques fonctions utilisées dans les premières sections. Elle suppose que le notebook est exécuté depuis le dossier `collazione/`.


In [13]:
from pathlib import Path
from html import escape
from IPython.display import display, HTML, FileLink
import ipywidgets as widgets
import re

from collatex import Collation, collate

try:
    from lxml import etree
except Exception:
    etree = None

# Le notebook est dans /collazione : on remonte donc d'un niveau.
BASE = Path("..")
DATA = BASE / "data"
TEI = DATA / "tei"
EXPORTS = Path("exports")
EXPORTS.mkdir(exist_ok=True)


def read_txt(path: Path) -> str:
    """Lit un témoin en texte brut."""
    return path.read_text(encoding="utf-8").strip()


def read_tei_text(path: Path) -> str:
    """Extrait le texte courant d'un fichier TEI.

    Pour une première collation, on ne garde pas ici les balises : CollateX
    reçoit une chaîne de caractères par témoin.
    """
    if etree is None:
        raise RuntimeError("La bibliothèque lxml n'est pas disponible.")

    parser = etree.XMLParser(recover=True, huge_tree=True)
    xml = etree.parse(str(path), parser)
    ns = {"tei": "http://www.tei-c.org/ns/1.0"}
    nodes = xml.xpath("//tei:body", namespaces=ns) or xml.xpath("//tei:text", namespaces=ns)
    if not nodes:
        raise ValueError(f"Aucun <body> ou <text> TEI trouvé dans {path}")

    text = " ".join(nodes[0].itertext())
    return re.sub(r"\s+", " ", text).strip()


def load_witnesses(corpus: str, fmt: str = "txt", normalized: bool = False) -> dict:
    """Charge tous les témoins d'un corpus.

    fmt='txt' lit ../data/<corpus>/*.txt
    fmt='tei' lit ../data/tei/<corpus>/*.xml
    """
    if fmt == "txt":
        folder = DATA / corpus / ("norm" if normalized else "")
        suffix = "*.txt"
        reader = read_txt
    elif fmt == "tei":
        folder = TEI / corpus
        suffix = "*.xml"
        reader = read_tei_text
    else:
        raise ValueError("fmt doit être 'txt' ou 'tei'.")

    if not folder.exists():
        raise FileNotFoundError(f"Dossier introuvable : {folder}")

    witnesses = {}
    for path in sorted(folder.glob(suffix)):
        siglum = path.stem.replace("_norm", "")
        witnesses[siglum] = reader(path)

    if not witnesses:
        raise FileNotFoundError(f"Aucun témoin {suffix} trouvé dans {folder}")
    return witnesses


def make_collation(witnesses: dict) -> Collation:
    collation = Collation()
    for siglum, text in witnesses.items():
        collation.add_plain_witness(siglum, text)
    return collation


def collate_to_html(witnesses: dict, segmentation: bool = True, layout: str = "vertical") -> str:
    return collate(make_collation(witnesses), output="html", layout=layout, segmentation=segmentation)


def collate_to_tei(witnesses: dict) -> str:
    return collate(make_collation(witnesses), output="tei")


def quick_stats(witnesses: dict):
    rows = []
    for siglum, text in witnesses.items():
        tokens = re.findall(r"\w+|[^\w\s]", text, flags=re.UNICODE)
        rows.append((siglum, len(text), len(tokens)))
    return sorted(rows)

print("Prêt : chemins chargés, fonctions disponibles.")
print("DATA =", DATA)
print("TEI  =", TEI)


Prêt : chemins chargés, fonctions disponibles.
DATA = ../data
TEI  = ../data/tei


## 1. Première collation : fichiers TXT

On commence par une collation très simple. Chaque témoin est un fichier `.txt`. CollateX aligne les chaînes de mots et signale les lieux où les témoins divergent.

Le réglage `segmentation=True` regroupe les passages identiques entre deux zones de variation. Pour une première lecture, c’est souvent plus lisible.


In [ ]:
corpus_dd = widgets.Dropdown(
    options=["poggio1", "poggio2", "dante1", "dante2"],
    value="poggio1",
    description="Corpus :",
    style={"description_width": "initial"},
)
norm_cb = widgets.Checkbox(
    value=False,
    description="Utiliser les versions normalisées, si elles existent"
)
seg_cb = widgets.Checkbox(
    value=True,
    description="Regrouper les passages non variants"
)

run_btn = widgets.Button(description="Collationner", button_style="success")
out = widgets.Output()


def run_txt_collation(_=None):
    out.clear_output()
    with out:
        corpus = corpus_dd.value
        normalized = norm_cb.value
        segmentation = seg_cb.value
        witnesses = load_witnesses(corpus, fmt="txt", normalized=normalized)

        display(HTML(f"<h3>Résultats — {escape(corpus)} (TXT)</h3>"))
        display(HTML("<p><b>Témoins chargés :</b> " + ", ".join(map(escape, witnesses.keys())) + "</p>"))

        rows = "".join(
            f"<tr><td>{escape(siglum)}</td><td>{nchars}</td><td>{ntokens}</td></tr>"
            for siglum, nchars, ntokens in quick_stats(witnesses)
        )
        display(HTML(
            "<h4>Contrôle rapide</h4>"
            "<table><tr><th>Témoin</th><th>Caractères</th><th>Tokens</th></tr>"
            + rows + "</table>"
        ))

        html = collate_to_html(witnesses, segmentation=segmentation)
        display(HTML(html))

run_btn.on_click(run_txt_collation)
display(widgets.VBox([widgets.HBox([corpus_dd, norm_cb, seg_cb, run_btn]), out]))


## 2. Export TEI-XML de la collation

La même collation peut être exportée en TEI. Cette sortie est moins agréable à lire à l’écran, mais elle est utile pour une édition critique, car elle encode les variantes dans une structure réutilisable.

Exécutez la cellule après avoir choisi le corpus dans la section précédente.


In [ ]:
corpus = corpus_dd.value
normalized = norm_cb.value
witnesses = load_witnesses(corpus, fmt="txt", normalized=normalized)

tei_xml = collate_to_tei(witnesses)
out_path = Path(f"collation_{corpus}_{'norm' if normalized else 'raw'}.xml")
out_path.write_text(tei_xml, encoding="utf-8")

display(HTML("<p>TEI-XML généré :</p>"))
display(FileLink(str(out_path)))
display(HTML(
    "<details><summary>Aperçu du début du fichier</summary><pre>"
    + escape(tei_xml[:1200])
    + "</pre></details>"
))


## 3. Collation de fichiers XML-TEI

On passe maintenant à des témoins XML-TEI. Pour rester dans la logique de CollateX, la cellule extrait d’abord le texte courant du `<body>`, puis l’envoie à CollateX comme dans la section précédente.

Cette étape est volontairement simple : elle montre la différence entre « lire un TEI » et « exploiter toute sa structure ».


In [ ]:
tei_corpus_dd = widgets.Dropdown(
    options=["poggio1", "poggio2", "dante1", "dante2", "cliges"],
    value="poggio1",
    description="Corpus TEI :",
    style={"description_width": "initial"},
)
tei_btn = widgets.Button(description="Collationner TEI", button_style="info")
out2 = widgets.Output()


def run_tei_collation(_=None):
    out2.clear_output()
    with out2:
        corpus = tei_corpus_dd.value
        witnesses = load_witnesses(corpus, fmt="tei")
        display(HTML(f"<h3>Résultats — {escape(corpus)} (XML-TEI)</h3>"))
        display(HTML("<p><b>Témoins chargés :</b> " + ", ".join(map(escape, witnesses.keys())) + "</p>"))
        display(HTML(collate_to_html(witnesses, segmentation=True)))

tei_btn.on_click(run_tei_collation)
display(widgets.VBox([widgets.HBox([tei_corpus_dd, tei_btn]), out2]))


## 4. Témoins lemmatisés : comparer des lemmes plutôt que des formes

Les fichiers `../data/tei/cliges_lemma/Cliges_A.xml` et `Cliges_P.xml` contiennent des tokens `<w>` avec des attributs `lemma` et `pos`. On ne cherche donc plus seulement à comparer les formes écrites, mais à repérer des différences lexicales ou grammaticales.

Le principe de la section est le suivant :

1. lire chaque vers TEI comme une liste de tokens ;
2. construire une petite « signature » de vers à partir des derniers lemmes lexicaux ;
3. aligner les vers des deux témoins ;
4. à l’intérieur des vers alignés, aligner les tokens ;
5. exporter les différences dans un tableau TSV.

La méthode reste heuristique : elle est utile pour explorer un corpus, pas pour remplacer la vérification philologique.


In [ ]:
TEI_NS = {"tei": "http://www.tei-c.org/ns/1.0"}


def norm_lemma(value: str) -> str:
    return (value or "").strip().lower()


def norm_pos(value: str) -> str:
    return (value or "").strip()


def is_punct_pos(pos: str) -> bool:
    pos = (pos or "").strip()
    return pos.startswith("PON") or pos.upper() in {"PUNCT", "PONCT"}


def parse_tei_lines(xml_path: Path):
    """Transforme un fichier TEI lemmatisé en liste de vers.

    Chaque vers garde ses tokens, mais aussi deux signatures utilisées pour
    l'alignement : une signature courte de fin de vers et une signature un peu
    plus large. Ces signatures évitent de dépendre uniquement des formes.
    """
    parser = etree.XMLParser(recover=True, huge_tree=True)
    tree = etree.parse(str(xml_path), parser)
    root = tree.getroot()

    lines = []
    for line_number, line_node in enumerate(root.xpath(".//tei:l", namespaces=TEI_NS), start=1):
        tokens = []
        for w in line_node.xpath(".//tei:w", namespaces=TEI_NS):
            tokens.append({
                "form": "".join(w.itertext()).strip(),
                "lemma": norm_lemma(w.get("lemma")),
                "pos": norm_pos(w.get("pos")),
            })

        lexical_lemmas = [
            token["lemma"]
            for token in tokens
            if token["lemma"] and not is_punct_pos(token["pos"])
        ]

        lines.append({
            "line_idx": line_number,
            "tokens": tokens,
            "sig_rhyme": lexical_lemmas[-2:],
            "sig_full": lexical_lemmas[-6:],
        })

    return lines


In [ ]:
def jaccard(a, b) -> float:
    """Similarité entre deux ensembles de lemmes."""
    A, B = set(a), set(b)
    if not A and not B:
        return 1.0
    if not A or not B:
        return 0.0
    return len(A & B) / len(A | B)


def line_similarity(line_a, line_b) -> float:
    """Score de ressemblance entre deux vers.

    La fin du vers compte davantage, car elle aide souvent à retrouver le bon
    alignement dans un texte versifié.
    """
    rhyme = jaccard(line_a["sig_rhyme"], line_b["sig_rhyme"])
    wider = jaccard(line_a["sig_full"], line_b["sig_full"])
    return 0.7 * rhyme + 0.3 * wider


def align_lines(lines_a, lines_b, gap_penalty: float = -0.45):
    """Alignement global des vers par programmation dynamique.

    Le résultat est une liste de paires (i, j). Si i ou j vaut None, cela
    signale un vers sans correspondant dans l'autre témoin.
    """
    n, m = len(lines_a), len(lines_b)
    dp = [[0.0] * (m + 1) for _ in range(n + 1)]
    back = [[None] * (m + 1) for _ in range(n + 1)]

    for i in range(1, n + 1):
        dp[i][0] = dp[i - 1][0] + gap_penalty
        back[i][0] = (i - 1, 0, "up")
    for j in range(1, m + 1):
        dp[0][j] = dp[0][j - 1] + gap_penalty
        back[0][j] = (0, j - 1, "left")

    for i in range(1, n + 1):
        for j in range(1, m + 1):
            choices = [
                (dp[i - 1][j - 1] + line_similarity(lines_a[i - 1], lines_b[j - 1]), i - 1, j - 1, "diag"),
                (dp[i - 1][j] + gap_penalty, i - 1, j, "up"),
                (dp[i][j - 1] + gap_penalty, i, j - 1, "left"),
            ]
            score, prev_i, prev_j, move = max(choices, key=lambda item: item[0])
            dp[i][j] = score
            back[i][j] = (prev_i, prev_j, move)

    pairs = []
    i, j = n, m
    while i > 0 or j > 0:
        prev_i, prev_j, move = back[i][j]
        if move == "diag":
            pairs.append((i - 1, j - 1))
        elif move == "up":
            pairs.append((i - 1, None))
        else:
            pairs.append((None, j - 1))
        i, j = prev_i, prev_j

    return list(reversed(pairs))


In [ ]:
def token_similarity(token_a, token_b) -> float:
    """Score de ressemblance entre deux tokens.

    Le lemme est prioritaire. La catégorie grammaticale sert surtout à éviter
    quelques alignements absurdes quand les lemmes diffèrent.
    """
    if not token_a or not token_b:
        return 0.0

    same_lemma = token_a["lemma"] and token_a["lemma"] == token_b["lemma"]
    same_pos = token_a["pos"] and token_a["pos"] == token_b["pos"]

    if same_lemma and same_pos:
        return 1.0
    if same_lemma:
        return 0.8
    if same_pos:
        return 0.25
    return 0.0


def align_tokens(tokens_a, tokens_b, gap_penalty: float = -0.6, ignore_punct: bool = True):
    """Aligne les tokens de deux vers déjà alignés."""
    if ignore_punct:
        tokens_a = [t for t in tokens_a if not is_punct_pos(t["pos"])]
        tokens_b = [t for t in tokens_b if not is_punct_pos(t["pos"])]

    n, m = len(tokens_a), len(tokens_b)
    dp = [[0.0] * (m + 1) for _ in range(n + 1)]
    back = [[None] * (m + 1) for _ in range(n + 1)]

    for i in range(1, n + 1):
        dp[i][0] = dp[i - 1][0] + gap_penalty
        back[i][0] = (i - 1, 0, "up")
    for j in range(1, m + 1):
        dp[0][j] = dp[0][j - 1] + gap_penalty
        back[0][j] = (0, j - 1, "left")

    for i in range(1, n + 1):
        for j in range(1, m + 1):
            choices = [
                (dp[i - 1][j - 1] + token_similarity(tokens_a[i - 1], tokens_b[j - 1]), i - 1, j - 1, "diag"),
                (dp[i - 1][j] + gap_penalty, i - 1, j, "up"),
                (dp[i][j - 1] + gap_penalty, i, j - 1, "left"),
            ]
            score, prev_i, prev_j, move = max(choices, key=lambda item: item[0])
            dp[i][j] = score
            back[i][j] = (prev_i, prev_j, move)

    pairs = []
    i, j = n, m
    while i > 0 or j > 0:
        prev_i, prev_j, move = back[i][j]
        if move == "diag":
            pairs.append((tokens_a[i - 1], tokens_b[j - 1]))
        elif move == "up":
            pairs.append((tokens_a[i - 1], None))
        else:
            pairs.append((None, tokens_b[j - 1]))
        i, j = prev_i, prev_j

    return list(reversed(pairs))


def detect_variants_in_line(tokens_a, tokens_b):
    """Classe les différences observées dans un couple de vers alignés."""
    variants = []
    for a, b in align_tokens(tokens_a, tokens_b, ignore_punct=True):
        if a is None and b is not None:
            variants.append(("ajout", None, b))
        elif b is None and a is not None:
            variants.append(("lacune", a, None))
        elif a["lemma"] != b["lemma"]:
            variants.append(("lexicale", a, b))
        elif a["pos"] != b["pos"]:
            variants.append(("pos_diff", a, b))
    return variants


In [ ]:
def extract_variants_between_two_tei(xml_a: Path, xml_b: Path, wit_a="A", wit_b="B"):
    """Pipeline complet : vers, tokens, puis tableau de variantes."""
    lines_a = parse_tei_lines(xml_a)
    lines_b = parse_tei_lines(xml_b)
    line_pairs = align_lines(lines_a, lines_b)

    variants = []
    for i, j in line_pairs:
        if i is None:
            variants.append({"type": "vers_ajoute", "lineA": None, "lineB": lines_b[j]["line_idx"], "witA": wit_a, "witB": wit_b})
            continue
        if j is None:
            variants.append({"type": "vers_absent", "lineA": lines_a[i]["line_idx"], "lineB": None, "witA": wit_a, "witB": wit_b})
            continue

        for kind, a, b in detect_variants_in_line(lines_a[i]["tokens"], lines_b[j]["tokens"]):
            variants.append({
                "type": kind,
                "lineA": lines_a[i]["line_idx"],
                "lineB": lines_b[j]["line_idx"],
                "witA": wit_a,
                "witB": wit_b,
                "A_form": None if a is None else a["form"],
                "A_lemma": None if a is None else a["lemma"],
                "A_pos": None if a is None else a["pos"],
                "B_form": None if b is None else b["form"],
                "B_lemma": None if b is None else b["lemma"],
                "B_pos": None if b is None else b["pos"],
            })
    return variants


In [ ]:
from collections import Counter, defaultdict

VARIANT_COLUMNS = [
    "type", "lineA", "lineB", "witA", "witB",
    "A_form", "A_lemma", "A_pos", "B_form", "B_lemma", "B_pos"
]


def export_variants_tsv(variants, out_path: Path):
    out_path.parent.mkdir(parents=True, exist_ok=True)
    with out_path.open("w", encoding="utf-8") as f:
        f.write("\t".join(VARIANT_COLUMNS) + "\n")
        for variant in variants:
            row = []
            for col in VARIANT_COLUMNS:
                value = variant.get(col, "")
                row.append("" if value is None else str(value).replace("\t", " ").replace("\n", " "))
            f.write("\t".join(row) + "\n")
    return out_path


def show_summary(variants):
    counts = Counter(v["type"] for v in variants)
    items = "".join(f"<li><b>{escape(kind)}</b> : {count}</li>" for kind, count in counts.most_common())
    display(HTML("<h3>Résumé des différences repérées</h3><ul>" + items + "</ul>"))


def show_variants_table(variants, limit=40, only_lexical=False):
    rows = [v for v in variants if (not only_lexical or v.get("type") == "lexicale")]
    if limit is not None:
        rows = rows[:limit]
    if not rows:
        display(HTML("<p>Aucune ligne à afficher.</p>"))
        return

    html = "<table style='border-collapse:collapse;'>"
    html += "<tr><th>type</th><th>vers A</th><th>vers B</th><th>A</th><th>lemme/POS A</th><th>B</th><th>lemme/POS B</th></tr>"
    style = "border:1px solid #999;padding:4px;"
    for v in rows:
        html += "<tr>"
        html += f"<td style='{style}'>{escape(str(v.get('type','')))}</td>"
        html += f"<td style='{style}'>{escape(str(v.get('lineA','')))}</td>"
        html += f"<td style='{style}'>{escape(str(v.get('lineB','')))}</td>"
        html += f"<td style='{style}'>{escape(str(v.get('A_form','')))}</td>"
        html += f"<td style='{style}'><b>{escape(str(v.get('A_lemma','')))}</b> / {escape(str(v.get('A_pos','')))}</td>"
        html += f"<td style='{style}'>{escape(str(v.get('B_form','')))}</td>"
        html += f"<td style='{style}'><b>{escape(str(v.get('B_lemma','')))}</b> / {escape(str(v.get('B_pos','')))}</td>"
        html += "</tr>"
    html += "</table>"
    display(HTML(html))


### 4.1 Lancer la comparaison de `Cliges_A` et `Cliges_P`

Cette cellule produit trois choses : un résumé, un aperçu des premières variantes, et un fichier TSV dans `collazione/exports/`.


In [ ]:
CORPUS_DIR = Path("../data/tei/cliges_lemma")
base = "Cliges_A"
other = "Cliges_P"

A_path = CORPUS_DIR / f"{base}.xml"
P_path = CORPUS_DIR / f"{other}.xml"

variants = extract_variants_between_two_tei(A_path, P_path, wit_a=base, wit_b=other)

show_summary(variants)
display(HTML("<h3>Aperçu des premières lignes du tableau</h3>"))
show_variants_table(variants, limit=40)

out_path = export_variants_tsv(variants, EXPORTS / "cliges_lemma_variants.tsv")
print("TSV exporté :", out_path)


### 4.2 Lire uniquement les variantes lexicales

Les lignes `ajout`, `lacune` et `pos_diff` peuvent être utiles, mais elles dispersent la lecture. La cellule suivante ne garde que les cas où le lemme diffère entre les deux témoins.


In [ ]:
lexical_variants = [v for v in variants if v.get("type") == "lexicale"]
print("Nombre de variantes lexicales :", len(lexical_variants))
show_variants_table(lexical_variants, limit=80)


## 5. Exploiter les résultats

À partir du tableau de variantes, on peut commencer à formuler des questions :

- quels types de mots varient le plus souvent ?
- les variantes sont-elles concentrées dans certaines zones du texte ?
- certaines substitutions lexicales reviennent-elles plusieurs fois ?
- les variantes apparaissent-elles dans des zones lexicalement plus denses ?

Les cellules suivantes donnent des instruments simples pour ouvrir cette discussion.


In [ ]:
# Répartition des variantes lexicales par catégorie grammaticale du témoin de base.
pos_counts = Counter(v["A_pos"] for v in lexical_variants)

html = "<h3>Variantes lexicales par POS</h3><ul>"
for pos, count in pos_counts.most_common():
    html += f"<li><b>{escape(pos)}</b> : {count}</li>"
html += "</ul>"
display(HTML(html))


In [ ]:
# Répartition du texte en dix segments de longueur comparable.
if lexical_variants:
    max_line = max(v["lineA"] for v in lexical_variants if v.get("lineA") is not None)
else:
    max_line = 0

bins = 10
segment_counts = [0] * bins

for variant in lexical_variants:
    line = variant.get("lineA")
    if line is None or max_line == 0:
        continue
    index = int((line - 1) / max_line * bins)
    segment_counts[min(index, bins - 1)] += 1

html = "<h3>Répartition des variantes lexicales dans le texte</h3><ol>"
for i, count in enumerate(segment_counts, start=1):
    html += f"<li>Segment {i} : {count} variantes</li>"
html += "</ol>"
display(HTML(html))


In [ ]:
# Substitutions récurrentes : même lemme de départ, même lemme d'arrivée.
pairs = Counter(
    (v["A_lemma"], v["B_lemma"])
    for v in lexical_variants
    if v.get("A_lemma") and v.get("B_lemma")
)

html = "<h3>Substitutions lexicales récurrentes</h3><ul>"
for (lemma_a, lemma_b), count in pairs.most_common(15):
    html += f"<li><b>{escape(lemma_a)}</b> → <b>{escape(lemma_b)}</b> : {count}</li>"
html += "</ul>"
display(HTML(html))


## 6. Indicateurs complémentaires

Les indicateurs qui suivent ne donnent pas une interprétation à eux seuls. Ils servent plutôt à repérer des passages où il vaut la peine de revenir au texte.


In [ ]:
from statistics import mean

with_both_lemmas = [
    v for v in lexical_variants
    if v.get("A_lemma") and v.get("B_lemma")
]

len_a = [len(v["A_lemma"]) for v in with_both_lemmas]
len_b = [len(v["B_lemma"]) for v in with_both_lemmas]
diff = [b - a for a, b in zip(len_a, len_b)]

print("Variantes lexicales avec deux lemmes :", len(with_both_lemmas))
print("Longueur moyenne du lemme dans A :", round(mean(len_a), 2) if len_a else "NA")
print("Longueur moyenne du lemme dans B :", round(mean(len_b), 2) if len_b else "NA")
print("Différence moyenne B - A :", round(mean(diff), 2) if diff else "NA")


In [ ]:
# Vers où plusieurs variantes lexicales se concentrent.
density = Counter(v["lineA"] for v in lexical_variants if v.get("lineA") is not None)

print("Vers les plus denses :")
for line, count in density.most_common(20):
    print(f"Vers {line:>5} : {count}")


In [ ]:
# Carte HTML simple : un rectangle par vers, intensité selon le nombre de variantes.
if not density:
    raise RuntimeError("Aucune variante lexicale avec numéro de vers.")

max_line_a = max(density.keys())
dens_vec = [density.get(i, 0) for i in range(1, max_line_a + 1)]
max_density = max(dens_vec) or 1
dens_norm = [value / max_density for value in dens_vec]


def red_scale(value):
    red = int(255 * value)
    return f"rgb(255,{255-red},{255-red})"

html = "<h3>Carte couleur — densité des variantes lexicales par vers</h3>"
html += "<p>Chaque rectangle correspond à un vers du témoin de base.</p>"
html += "<div style='display:flex; flex-wrap:wrap; max-width:900px;'>"
for i, value in enumerate(dens_norm, start=1):
    html += (
        f"<div title='Vers {i} : {dens_vec[i-1]} variante(s)' "
        f"style='width:10px;height:18px;border:1px solid #eee;background:{red_scale(value)}'></div>"
    )
html += "</div>"
display(HTML(html))


In [ ]:
# Diversité lexicale locale autour des vers variant : TTR dans une fenêtre de ±5 vers.
lines_a = parse_tei_lines(A_path)


def lemmas_of_line(line, ignore_punct=True):
    lemmas = []
    for token in line["tokens"]:
        if not token["lemma"]:
            continue
        if ignore_punct and is_punct_pos(token["pos"]):
            continue
        lemmas.append(token["lemma"])
    return lemmas

lemmas_by_line = {line["line_idx"]: lemmas_of_line(line) for line in lines_a}


def local_ttr(center_line: int, window: int = 5) -> float:
    lemmas = []
    for line_number in range(center_line - window, center_line + window + 1):
        lemmas.extend(lemmas_by_line.get(line_number, []))
    if not lemmas:
        return 0.0
    return len(set(lemmas)) / len(lemmas)

ttr_scores = {line: local_ttr(line, window=5) for line in sorted(density)}

print("Top 15 des zones à TTR local élevé parmi les vers variant :")
for line, score in sorted(ttr_scores.items(), key=lambda item: -item[1])[:15]:
    print(f"Vers {line:>5} | TTR±5 = {score:.3f} | densité = {density[line]}")


### Comment lire le TTR local ?

Le TTR est le rapport entre le nombre de lemmes distincts et le nombre total de lemmes dans une fenêtre donnée. Ici, on regarde une fenêtre de cinq vers avant et cinq vers après le vers variant.

Un TTR élevé signale une zone où les répétitions lexicales sont moins nombreuses. Cela ne veut pas dire automatiquement que la variante est plus importante, mais cela peut attirer l’attention sur des passages narratifs ou descriptifs plus denses.


In [ ]:
# Afficher les vers les plus denses, avec mise en évidence des lemmes impliqués dans une variante.
TOP_N = 10
WINDOW = 0

lines_by_idx = {line["line_idx"]: line for line in lines_a}
variant_lemmas_by_line = defaultdict(set)
for variant in lexical_variants:
    if variant.get("lineA") is not None and variant.get("A_lemma"):
        variant_lemmas_by_line[variant["lineA"]].add(variant["A_lemma"])


def render_line(line):
    pieces = []
    marked = variant_lemmas_by_line.get(line["line_idx"], set())
    for token in line["tokens"]:
        form = escape(token["form"])
        if token["lemma"] in marked:
            pieces.append(f"<b>{form}</b>")
        else:
            pieces.append(form)
    return " ".join(pieces)


def line_score(line_number):
    return (density.get(line_number, 0), ttr_scores.get(line_number, 0.0))

selected_lines = sorted(density.keys(), key=line_score, reverse=True)[:TOP_N]

html = "<h3>Vers à examiner en priorité</h3>"
for line_number in selected_lines:
    html += f"<hr><p><b>Vers {line_number}</b> — densité={density[line_number]}"
    html += f", TTR±5={ttr_scores.get(line_number, 0.0):.3f}</p>"
    for context_line in range(line_number - WINDOW, line_number + WINDOW + 1):
        if context_line in lines_by_idx:
            prefix = "➤ " if context_line == line_number else "&nbsp;&nbsp;"
            html += f"<p>{prefix}{render_line(lines_by_idx[context_line])}</p>"

display(HTML(html))


In [ ]:
# Exporter une carte HTML imprimable.
def blue_scale(value):
    blue = int(255 * value)
    return f"rgb({255-blue},{255-blue},255)"

max_ttr = max(ttr_scores.values()) if ttr_scores else 1
all_ttr = [ttr_scores.get(i, 0.0) / max_ttr for i in range(1, max_line_a + 1)]

html = """<!doctype html>
<html lang="fr">
<head>
<meta charset="utf-8">
<title>Carte couleur — variantes lexicales</title>
<style>
  body { font-family: system-ui, -apple-system, Segoe UI, Roboto, Arial, sans-serif; margin: 24px; }
  .row { display:flex; flex-wrap:wrap; max-width: 980px; margin-bottom: 18px; }
  .cell { width: 10px; height: 20px; border: 1px solid #eee; }
</style>
</head>
<body>
<h2>Carte couleur — variantes lexicales</h2>
<p>Ligne rouge : densité des variantes lexicales. Ligne bleue : TTR local, quand disponible.</p>
<h3>Densité</h3>
<div class="row">
"""

for i, value in enumerate(dens_norm, start=1):
    html += f"<div class='cell' title='Vers {i} : {dens_vec[i-1]} variante(s)' style='background:{red_scale(value)}'></div>\n"

html += "</div><h3>TTR local</h3><div class='row'>\n"
for i, value in enumerate(all_ttr, start=1):
    html += f"<div class='cell' title='Vers {i} : TTR normalisé {value:.3f}' style='background:{blue_scale(value)}'></div>\n"

html += """
</div>
<p><small>Pour sauvegarder en PDF : imprimer depuis le navigateur, puis choisir « Enregistrer en PDF ».</small></p>
</body></html>
"""

html_path = EXPORTS / "carte_variantes.html"
html_path.write_text(html, encoding="utf-8")
display(HTML(f"<p>Carte HTML exportée : <code>{html_path}</code></p>"))


In [ ]:
# Export SVG vectoriel de la carte de densité.
cell_w, cell_h = 8, 18
margin = 20
width = margin * 2 + cell_w * max_line_a
height = margin * 2 + cell_h

svg = [f"""<svg xmlns=\"http://www.w3.org/2000/svg\" width=\"{width}\" height=\"{height}\">
<rect x=\"0\" y=\"0\" width=\"{width}\" height=\"{height}\" fill=\"white\"/>
<text x=\"{margin}\" y=\"{margin-6}\" font-family=\"Arial\" font-size=\"12\">Densité des variantes lexicales par vers</text>
"""]

for i, value in enumerate(dens_norm):
    x = margin + i * cell_w
    svg.append(f'<rect x="{x}" y="{margin}" width="{cell_w}" height="{cell_h}" fill="{red_scale(value)}" stroke="#eee"/>')

svg.append("</svg>")
svg_path = EXPORTS / "carte_variantes.svg"
svg_path.write_text("\n".join(svg), encoding="utf-8")
display(HTML(f"<p>SVG exporté : <code>{svg_path}</code></p>"))
